In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
import joblib
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner


data = pd.read_csv('data.csv')
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

test_subjects = [11, 12, 13]
train_df = data[~data['subject'].isin(test_subjects)].copy()
val_df = data[data['subject'].isin(test_subjects)].copy()


landmark_names = [
    'nose', 'left_eye_inner', 'left_eye', 'left_eye_outer',
    'right_eye_inner', 'right_eye', 'right_eye_outer',
    'left_ear', 'right_ear',
    'mouth_left', 'mouth_right',
    'left_shoulder', 'right_shoulder',
    'left_elbow', 'right_elbow'
]

upper_cols = []
for name in landmark_names:
    upper_cols.extend([f'{name}_x', f'{name}_y', f'{name}_z'])


def compute_features(pts):
    nose = pts[0]
    left_shoulder = pts[11]
    right_shoulder = pts[12]
    left_elbow = pts[13]
    right_elbow = pts[14]

    shoulder_mid = (left_shoulder + right_shoulder) / 2.0
    shoulder_width = np.linalg.norm(left_shoulder - right_shoulder)
    if shoulder_width < 1e-6:
        shoulder_width = 1.0

    f = []

    vec = nose - shoulder_mid
    vertical = np.array([0, 1, 0])
    cos_angle = np.dot(vec, vertical) / (np.linalg.norm(vec) * np.linalg.norm(vertical) + 1e-8)
    f.append(np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0))))

    shoulder_vec = right_shoulder - left_shoulder
    horizontal = np.array([1, 0, 0])
    cos_slope = np.dot(shoulder_vec, horizontal) / (np.linalg.norm(shoulder_vec) * np.linalg.norm(horizontal) + 1e-8)
    f.append(np.degrees(np.arccos(np.clip(cos_slope, -1.0, 1.0))))

    f.append(np.linalg.norm(nose - shoulder_mid) / shoulder_width)

    f.append((left_elbow[2] - left_shoulder[2]) / shoulder_width)
    f.append((left_elbow[0] - left_shoulder[0]) / shoulder_width)
    f.append((left_elbow[1] - left_shoulder[1]) / shoulder_width)
    f.append(np.linalg.norm(left_elbow - left_shoulder) / shoulder_width)

    f.append((right_elbow[2] - right_shoulder[2]) / shoulder_width)
    f.append((right_elbow[0] - right_shoulder[0]) / shoulder_width)
    f.append((right_elbow[1] - right_shoulder[1]) / shoulder_width)
    f.append(np.linalg.norm(right_elbow - right_shoulder) / shoulder_width)

    f.append((left_shoulder[1] - right_shoulder[1]) / shoulder_width)

    vec_proj = vec.copy()
    vec_proj[1] = 0
    shoulder_proj = shoulder_vec.copy()
    shoulder_proj[1] = 0
    if np.linalg.norm(vec_proj) > 0 and np.linalg.norm(shoulder_proj) > 0:
        cos_rot = np.dot(vec_proj, shoulder_proj) / (np.linalg.norm(vec_proj) * np.linalg.norm(shoulder_proj) + 1e-8)
        f.append(np.degrees(np.arccos(np.clip(cos_rot, -1.0, 1.0))))
    else:
        f.append(0.0)

    f.append(np.linalg.norm(left_elbow - right_elbow) / shoulder_width)

    return np.array(f)


def extract_features(df):
    X = df[upper_cols].values.reshape(-1, len(landmark_names), 3)
    return np.array([compute_features(pts) for pts in X])


X_train_raw = extract_features(train_df)
X_val_raw = extract_features(val_df)

y_train = (train_df['upperbody_label'] != 'TLB').astype(int)
y_val = (val_df['upperbody_label'] != 'TLB').astype(int)


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val = scaler.transform(X_val_raw)


def augment(X, y, num_copies=2, noise_std=0.05):
    X_aug, y_aug = X.copy(), y.copy()
    for _ in range(num_copies - 1):
        X_aug = np.vstack([X_aug, X + np.random.normal(0, noise_std, X.shape)])
        y_aug = np.hstack([y_aug, y])
    return X_aug, y_aug


X_train_aug, y_train_aug = augment(X_train, y_train, num_copies=2)


def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300, step=50),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'class_weight': trial.suggest_categorical('class_weight', ['balanced', 'balanced_subsample', None]),
        'random_state': 42,
        'n_jobs': -1
    }
    clf = RandomForestClassifier(**params)
    clf.fit(X_train_aug, y_train_aug)
    preds = clf.predict(X_val)
    score = f1_score(y_val, preds, average='weighted')
    return score


study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=10)
)
study.optimize(objective, n_trials=30, timeout=600)  # 30 trials or 10 minutes


best_params = study.best_params
print("Best parameters:", best_params)
print("Best validation F1:", study.best_value)


best_clf = RandomForestClassifier(
    **best_params,
    random_state=42,
    n_jobs=-1
)
best_clf.fit(X_train_aug, y_train_aug)

final_preds = best_clf.predict(X_val)
final_score = f1_score(y_val, final_preds, average='weighted')
print(f"Final validation F1 with best params: {final_score:.4f}")


joblib.dump(best_clf, 'posture_model_optuna.pkl')
joblib.dump(scaler, 'scaler_optuna.pkl')

[I 2026-06-16 19:13:46,970] A new study created in memory with name: no-name-801f7c8f-1030-4fbb-bb89-352eb93b7e20
[I 2026-06-16 19:13:48,102] Trial 0 finished with value: 0.8941659716969544 and parameters: {'n_estimators': 150, 'max_depth': 20, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': 'sqrt', 'class_weight': 'balanced'}. Best is trial 0 with value: 0.8941659716969544.
[I 2026-06-16 19:13:49,535] Trial 1 finished with value: 0.8804895325558528 and parameters: {'n_estimators': 50, 'max_depth': 20, 'min_samples_split': 17, 'min_samples_leaf': 3, 'max_features': None, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.8941659716969544.
[I 2026-06-16 19:13:50,906] Trial 2 finished with value: 0.8961362597165067 and parameters: {'n_estimators': 200, 'max_depth': 7, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': 'log2', 'class_weight': 'balanced_subsample'}. Best is trial 2 with value: 0.8961362597165067.
[I 2026-06-16 19:13:52,249] Trial 3 finis

Best parameters: {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 14, 'min_samples_leaf': 7, 'max_features': 'log2', 'class_weight': 'balanced_subsample'}
Best validation F1: 0.9162778519260001
Final validation F1 with best params: 0.9163


['scaler_optuna.pkl']